## Import and install dependencies

In [9]:
%pip install evaluate transformers torch accelerate -q

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)

cuda


In [11]:
BASE_MODEL_NAME = "prajjwal1/bert-tiny"

model = AutoModel.from_pretrained(BASE_MODEL_NAME)

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

BertModel LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
model.config

BertConfig {
  "add_cross_attention": false,
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 128,
  "initializer_range": 0.02,
  "intermediate_size": 512,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 2,
  "num_hidden_layers": 2,
  "pad_token_id": 0,
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

## Loading Dataset

In [13]:
from datasets import Dataset, DatasetDict, load_dataset

SUBSET_SIZE = 1000

pt_stream = load_dataset('bastao/VeraCruz_PT-BR', "Portugal (PT)", split='train', streaming=True)
ptbr_stream = load_dataset('bastao/VeraCruz_PT-BR', "Brazil (BR)", split='train', streaming=True)
other_stream = load_dataset('bastao/VeraCruz_PT-BR', "Other", split='train', streaming=True)

pt_subset = pt_stream.take(SUBSET_SIZE)
ptbr_subset = ptbr_stream.take(SUBSET_SIZE)
other_subset = other_stream.take(SUBSET_SIZE)

def stream_to_dataset(stream):
    return Dataset.from_list(list(stream))

VERA_CRUZ_DATASETS = DatasetDict({
    "pt": stream_to_dataset(pt_subset),
    "ptbr": stream_to_dataset(ptbr_subset),
    "other": stream_to_dataset(other_subset),
})

VERA_CRUZ_DATASETS

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

## Tokenize

In [16]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

def preprocess(sample):
    return tokenizer(sample['text'], truncation=True)

TOKENIZED_VERA_CRUZ_DATASETS = VERA_CRUZ_DATASETS.map(preprocess, batched=True)

TOKENIZED_VERA_CRUZ_DATASETS

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    pt: Dataset({
        features: ['text', 'timestamp', 'url', 'source', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1000
    })
    ptbr: Dataset({
        features: ['text', 'timestamp', 'url', 'source', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1000
    })
    other: Dataset({
        features: ['text', 'timestamp', 'url', 'source', 'label', 'score', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1000
    })
})

In [21]:
TOKENIZED_VERA_CRUZ_DATASETS["other"]

Dataset({
    features: ['text', 'timestamp', 'url', 'source', 'label', 'score', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1000
})

## Fine-tuning

In [24]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

# data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir='teste',
    per_device_train_batch_size=64,
    logging_strategy='epoch',
    num_train_epochs=3,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=TOKENIZED_VERA_CRUZ_DATASETS['other'],
    # eval_dataset=tokenized_imdb["test"],
    # data_collator=data_collator,
    # compute_metrics=compute_metrics,
)


In [25]:
trainer.train()

ValueError: too many dimensions 'str'

## Evaluation

In [ ]:
# from sklearn.metrics import accuracy_score

# def compute_metrics(pred):
#     labels = pred.label_ids
#     preds = pred.predictions.argmax(-1)
#     return accuracy_score(labels, preds)

# preds = trainer.predict(TOKENIZED_VERA_CRUZ_DATASETS['test'])
# compute_metrics(preds)